# Lab 7 - 通常の Agent から Harness Agent へ

この Notebook では Microsoft Agent Framework が初めてでも追えるように、最初に **Foundry IQ だけを使う通常の Agent** を作ります。次に同じ Foundry IQ を残し、Lab 4 の Toolbox tools と Skills、plan / execute mode、todos、bounded loop を加えた **Harness Agent** へ発展させます。

**使用する kernel:** `Python (Foundry Hosted Agent)`。Codespaces またはローカルの同じ Dev Container で `00-setup.ipynb` を完了してから開きます。コンテナーの再起動後は kernel 内の session が復元されないため、接続先の確認からやり直してください。

| 段階 | 作るもの | 観察すること |
|---|---|---|
| 1 | `Agent(...)` + Foundry IQ | client / instructions / tool / run の最小構成 |
| 2 | `create_harness_agent(...)` + Foundry IQ + Toolbox + Skills | task decomposition、Skill の遅延読み込み、Tool Search、複数 tool、todos の完了 |

> **データ境界と料金:** コードは Notebook で動きますが、モデル、Foundry IQ、Toolbox、Code Interpreter、Web Search は Azure 上で実行されます。合成データだけを使い、secret・個人情報・顧客情報を入力しないでください。同じ実行 cell を結果待ちの間に再実行しないでください。

## 1. Lab 1〜4 の接続先を読み込む

`00-setup.ipynb` が生成した `.workshop/context.json` から project、model、Azure AI Search の endpoint を取得します。Foundry IQ と Toolbox の名前は Lab 3 / 4 で作成した固定名です。API key や client secret は読みません。

In [ ]:
import os
import sys
from pathlib import Path


def find_repo_root() -> Path:
    current = Path.cwd().resolve()
    for candidate in (current, *current.parents):
        if (candidate / "src" / "hosted-agent" / "travel_agents.py").is_file():
            return candidate
    raise RuntimeError(
        "Repository root が見つかりません。clone した教材内の Notebook を開いてください。"
    )


REPO_ROOT = find_repo_root()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from scripts.lib.workshop_context import load_context, validate_workshop_context  # noqa: E402
from scripts.lib.workshop_runtime import HOSTED_ENVIRONMENT, require_current_runtime  # noqa: E402

require_current_runtime(HOSTED_ENVIRONMENT)
context_path = REPO_ROOT / ".workshop" / "context.json"
if not context_path.is_file():
    raise FileNotFoundError("00-setup.ipynb を完了し、.workshop/context.json を作成してください。")

context = validate_workshop_context(load_context(context_path))
outputs = context["resource_outputs"]
os.environ["WORKSHOP_CREDENTIAL_MODE"] = "azure-cli"
os.environ["FOUNDRY_PROJECT_ENDPOINT"] = outputs["foundry_project_endpoint"]["value"]
os.environ["FOUNDRY_MODEL"] = outputs["primary_model_deployment_name"]["value"]
os.environ["AZURE_AI_SEARCH_SERVICE_ENDPOINT"] = outputs["search_service_endpoint"]["value"]
os.environ["AZURE_AI_SEARCH_KNOWLEDGE_BASE_NAME"] = "contoso-travel-knowledge-lab"
os.environ["TOOLBOX_NAME"] = "contoso-travel-toolbox"

hosted_source = REPO_ROOT / "src" / "hosted-agent"
if str(hosted_source) not in sys.path:
    sys.path.insert(0, str(hosted_source))

print(f"Project: {outputs['foundry_project_name']['value']}")
print(f"Model: {os.environ['FOUNDRY_MODEL']}")
print(f"Knowledge base: {os.environ['AZURE_AI_SEARCH_KNOWLEDGE_BASE_NAME']}")
print(f"Toolbox: {os.environ['TOOLBOX_NAME']}")

## 2. Foundry IQ を使う通常の Agent を作る

Agent の最小構成は、model を呼ぶ **client**、担当を決める **instructions**、外部機能の **tools** です。`Agent(...)` は Portal に resource を登録する操作ではなく、この Python プロセス内に実行可能な Agent object を作ります。

Foundry IQ の knowledge base は MCP endpoint として接続します。HTTP client は `az login` の token を毎回付け、`knowledge_base_retrieve` だけを公開します。

In [ ]:
import travel_agents

credential = travel_agents.create_credential()
plain_chat_client = travel_agents.create_chat_client(credential)
print(type(plain_chat_client).__name__)

In [ ]:
plain_question = (
    "片道12時間の国際線を出発2日前にビジネスクラスで予約したいです。"
    "直前予約の添付物、承認者と順序、申請機能名、標準最大営業日数を、"
    "根拠付きでまとめてください。"
)

plain_iq_tool = travel_agents.create_foundry_iq_tool(credential)
plain_agent = travel_agents.build_plain_travel_agent(
    chat_client=plain_chat_client,
    foundry_iq_tool=plain_iq_tool,
)

async with plain_agent:
    plain_response = await plain_agent.run(plain_question)

print(plain_response.text)

回答で 2 つの規程の citation を確認してください。この Agent に渡した tool は Foundry IQ だけです。費用計算や現在情報を尋ねても、持っていない tool を実行できません。

| コード | 意味 |
|---|---|
| `create_chat_client(...)` | Foundry の model を呼ぶ |
| `create_foundry_iq_tool(...)` | Lab 3 の knowledge base を MCP tool にする |
| `build_plain_travel_agent(...)` | client + instructions + tool を通常の `Agent` にまとめる |
| `agent.run(...)` | 1 回の依頼を実行する |

## 3. 同じ Agent に Harness の実行支援を追加する

`create_harness_agent` は通常の `Agent` を返す factory です。この例では次を追加します。

- plan / execute mode
- session に保存される todos
- `.workshop/harness-memory` に限定した file memory
- Toolbox Skills の progressive disclosure (`load_skill`)
- 未完了 todo がある execute mode だけを再実行する bounded loop

Harness が client から自動追加できる Web Search は無効にします。Web Search も Lab 4 の Toolbox と Tool Search を通して使うためです。

In [ ]:
from contextlib import AsyncExitStack

from agent_framework import (
    AgentModeProvider,
    FileSystemAgentFileStore,
    InMemoryHistoryProvider,
    TodoProvider,
    create_harness_agent,
    get_agent_mode,
    set_agent_mode,
    todos_remaining,
    todos_remaining_message,
)

harness_chat_client = travel_agents.create_chat_client(credential)
harness_iq_tool = travel_agents.create_foundry_iq_tool(credential)
toolbox = travel_agents.create_toolbox(credential)
skills_provider = toolbox.as_skills_provider(
    disable_load_skill_approval=True,
    disable_read_skill_resource_approval=True,
)
todo_provider = TodoProvider()
mode_provider = AgentModeProvider(default_mode="plan")
memory_store = FileSystemAgentFileStore(str(REPO_ROOT / ".workshop" / "harness-memory"))

harness_agent = create_harness_agent(
    client=harness_chat_client,
    name=travel_agents.HARNESS_AGENT_NAME,
    description=travel_agents.HARNESS_AGENT_DESCRIPTION,
    agent_instructions=travel_agents.HARNESS_AGENT_INSTRUCTIONS,
    tools=[harness_iq_tool, toolbox],
    history_provider=InMemoryHistoryProvider(),
    max_context_window_tokens=128_000,
    max_output_tokens=16_384,
    todo_provider=todo_provider,
    mode_provider=mode_provider,
    file_memory_store=memory_store,
    skills_provider=skills_provider,
    disable_web_search=True,
    disable_tool_auto_approval=True,
    loop_should_continue=todos_remaining(looping_modes=["execute"]),
    loop_next_message=todos_remaining_message,
    loop_max_iterations=6,
)

resource_stack = AsyncExitStack()
await resource_stack.__aenter__()
await resource_stack.enter_async_context(harness_agent)
harness_session = harness_agent.create_session()
print(type(harness_agent).__name__)
print(f"Mode: {get_agent_mode(harness_session)}")

### 3-1. 回答と tool call を観察する helper

`AgentResponse.messages` には最終文章だけでなく、モデルが要求した function/MCP call と結果も含まれます。次の helper は content の型と tool 名を一覧にし、回答を書き換えずに観察します。

In [ ]:
from typing import Any


def observed_actions(response: Any) -> list[str]:
    actions = []
    for message in response.messages:
        for content in message.contents:
            name = getattr(content, "name", None)
            if name:
                actions.append(str(name))
    return actions


def current_todos(session: Any) -> list[dict[str, Any]]:
    state = session.state.get(todo_provider.source_id, {})
    return list(state.get("items", [])) if isinstance(state, dict) else []

## 4. plan mode で複雑な依頼を todos に分解する

規程、API、計算、現在情報、承認シミュレーションを 1 回の依頼に含めます。最初は plan mode なので、Harness Agent は実行計画と todos を作り、承認を待ちます。

In [ ]:
from datetime import date, timedelta

start_date = date.today() + timedelta(days=30)
end_date = start_date + timedelta(days=2)
complex_request = f"""
東京からニューヨークへ {start_date.isoformat()} から {end_date.isoformat()} まで、
1名、business class で顧客ワークショップに行く想定です。予算は500,000円です。
社内規程と承認手続きを根拠付きで確認し、Travel Ops API の費用見積もり、
予算との差額と消化率、現在公開されているニューヨーク渡航上の注意情報を1件、
事前承認シミュレーションをまとめてください。現在情報には取得時点と出典を付け、
実際の予約や承認は行わないでください。
""".strip()
print(complex_request)

In [ ]:
plan_response = await harness_agent.run(complex_request, session=harness_session)
print(plan_response.text)
print("\nObserved actions:", observed_actions(plan_response))
print("\nTodos:")
for item in current_todos(harness_session):
    state = "done" if item.get("is_complete") else "open"
    print(f"- [{state}] {item.get('title') or item.get('description')}")

assert get_agent_mode(harness_session) == "plan"
assert current_todos(harness_session), (
    "todo が作成されませんでした。plan の出力を確認してください。"
)

## 5. 計画を承認し、同じ session を execute mode にする

計画に、実際の予約・承認や不要なデータ送信がないことを確認してから次を実行します。`set_agent_mode` は session 内の mode だけを変更します。plan mode で作った todos と会話は同じ session に残ります。

execute mode では未完了 todo が残る場合だけ loop します。`loop_max_iterations=6` が安全上限です。

In [ ]:
set_agent_mode(
    harness_session,
    "execute",
    source_id=mode_provider.source_id,
    available_modes=mode_provider.available_modes,
)
execute_response = await harness_agent.run(
    "この計画を承認します。todos を順に実行し、完了した項目を更新してください。",
    session=harness_session,
)
print(execute_response.text)
execute_actions = observed_actions(execute_response)
print("\nObserved actions:")
for action in execute_actions:
    print(f"- {action}")

todos_after_execute = current_todos(harness_session)
print("\nTodos after execute:")
for item in todos_after_execute:
    state = "done" if item.get("is_complete") else "open"
    print(f"- [{state}] {item.get('title') or item.get('description')}")

assert get_agent_mode(harness_session) == "execute"
assert todos_after_execute
assert all(item.get("is_complete") is True for item in todos_after_execute), todos_after_execute

## 6. 「なぜ複雑な問いを解けたか」を確認する

Observed actions と Portal の Trace を照合し、次の層を区別してください。

| 層 | 期待する記録 | 役割 |
|---|---|---|
| Harness | `todos_add` / `todos_complete`、mode | 依頼の分解と進捗管理 |
| Skills | `load_skill`、skill resource read | 必要な操作手順だけを遅延読み込み |
| Tool Search | `tool_search` → `call_tool` | 必要な tool schema を動的に発見・実行 |
| Foundry IQ | `knowledge_base_retrieve` | 社内規程と承認手続きの根拠 |
| Toolbox tools | Travel Ops / Code Interpreter / Web Search | API 結果、計算、明示的に求めた現在情報 |

Tool Search は task decomposition ではありません。Harness の mode / todos / session / loop が作業を管理し、Tool Search は各 todo に必要な tool を探します。Web Search の内容は変動するため、固定文ではなく取得時点・出典・失敗時の明示を確認します。

In [ ]:
import inspect

from IPython.display import Code, display

display(Code(inspect.getsource(travel_agents.build_harness_travel_agent), language="python"))
print("Lab 8 はこの Harness を引き継がず、通常 Agent の sequential workflow と比較します。")

## 7. Notebook の接続を閉じる

Toolbox と Foundry IQ の MCP session、HTTP client を明示的に閉じます。Azure resource は削除しません。resource の cleanup は Lab 9 で行います。

In [ ]:
await resource_stack.aclose()
credential.close()
print("Notebook resources closed.")

## 次の Lab

[Lab 8](../labs/08-hosted-multi-agent.md) では、Luna の token 消費を抑えるため Harness を引き継がず、intake / policy / reviewer の通常 Agent を sequential workflow として Hosted Agent に deploy します。Lab 8 はこの Notebook の session や出力には依存しないため、経験者は Lab 7 を飛ばしても実行できます。